# SAR-to-EO Image Translation — GalaxEye Assignment
**Model:** Pix2Pix U-Net + PatchGAN  
**Loss:** L1 + Adversarial + FFT Frequency + VGG Perceptual  
**Dataset:** Sentinel-1&2 terrain-split (Kaggle)  

**Checkpoints → saved directly to Google Drive (never lost)**


In [ ]:
# ================================================================
# CELL 1: Mount Google Drive + Check GPU
# Run this FIRST every session. Drive mount = checkpoints are safe.
# ================================================================
from google.colab import drive
drive.mount('/content/drive')

import torch, subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

import os
# Create persistent folder structure on Drive
DRIVE_DIR = '/content/drive/MyDrive/sar2eo'
for d in ['checkpoints', 'outputs', 'data']:
    os.makedirs(f'{DRIVE_DIR}/{d}', exist_ok=True)
print(f'\nDrive folder ready: {DRIVE_DIR}')

In [ ]:
# ================================================================
# CELL 2: Install packages
# ================================================================
!pip install lpips pytorch-fid scikit-image pyyaml -q

In [ ]:
# ================================================================
# CELL 3: Clone repo + set working directory
# ================================================================
import os, shutil, subprocess

REPO_DIR = '/content/sar2eo'
REPO_URL = 'https://github.com/Trafalgar-2006/sar2eo.git'

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

In [ ]:
# ================================================================
# CELL 4: Download dataset (FIRST TIME ONLY)
#
# Run this cell only ONCE to download the Sentinel dataset to Drive.
# After that, skip this cell — data is already on Drive.
#
# You need your Kaggle API key (kaggle.json):
#   kaggle.com → Account → API → Create New Token → download kaggle.json
# ================================================================
from google.colab import files
import os, shutil

DATASET_DRIVE = '/content/drive/MyDrive/sar2eo/data/sentinel12'

if os.path.exists(DATASET_DRIVE) and any(os.listdir(DATASET_DRIVE)):
    print(f'Dataset already on Drive at {DATASET_DRIVE} — skipping download.')
else:
    print('Upload your kaggle.json now...')
    files.upload()  # Upload kaggle.json when prompted

    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    shutil.copy('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

    print('Downloading dataset (~3GB)...')
    os.makedirs(DATASET_DRIVE, exist_ok=True)
    !kaggle datasets download -d requiemonk/sentinel12-image-pairs-segregated-by-terrain --unzip -p {DATASET_DRIVE}
    print(f'Done! Dataset saved to {DATASET_DRIVE}')

# Verify
terrains = [d for d in os.listdir(DATASET_DRIVE) if os.path.isdir(f'{DATASET_DRIVE}/{d}')]
print(f'Terrain folders found: {terrains}')

In [ ]:
# ================================================================
# CELL 5: Configure paths
# Checkpoints + outputs go DIRECTLY to Google Drive.
# ================================================================
import yaml, os

DRIVE_DIR    = '/content/drive/MyDrive/sar2eo'
DATASET_PATH = f'{DRIVE_DIR}/data/sentinel12'

assert os.path.exists(DATASET_PATH), f'Dataset not found at {DATASET_PATH}. Run Cell 4 first.'

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

# Data settings
cfg['data']['dataset_type']    = 'kaggle'
cfg['data']['kaggle_root']     = DATASET_PATH
cfg['data']['train_terrain']   = ['agri', 'barrenland', 'grassland']
cfg['data']['val_terrain']     = ['urban']
cfg['data']['test_terrain']    = ['urban']
cfg['data']['subset_size']     = None

# Training settings
cfg['training']['batch_size']  = 4
cfg['training']['epochs']      = 100
cfg['training']['num_workers'] = 2

# CRITICAL: save checkpoints + outputs directly to Google Drive
cfg['paths']['checkpoint_dir'] = f'{DRIVE_DIR}/checkpoints'
cfg['paths']['output_dir']     = f'{DRIVE_DIR}/outputs'

with open('config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('Config saved!')
print(f'  dataset       : {DATASET_PATH}')
print(f'  checkpoints   : {DRIVE_DIR}/checkpoints  ← saved to Drive')
print(f'  outputs       : {DRIVE_DIR}/outputs      ← saved to Drive')
print(f'  batch_size    : {cfg["training"]["batch_size"]}')
print(f'  epochs        : {cfg["training"]["epochs"]}')

In [ ]:
# ================================================================
# CELL 6: Smoke test (FIRST TIME ONLY — skip when resuming)
# ================================================================
import yaml, subprocess, copy

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg_s = copy.deepcopy(cfg)
cfg_s['data']['subset_size']   = 100
cfg_s['training']['epochs']    = 2
cfg_s['training']['val_freq']  = 1
cfg_s['training']['save_freq'] = 2
cfg_s['active_ablation']       = 'full'
# Smoke test saves locally (not Drive) to avoid clutter
cfg_s['paths']['checkpoint_dir'] = '/content/smoke_ckpts'
cfg_s['paths']['output_dir']     = '/content/smoke_outputs'

with open('config_smoke.yaml', 'w') as f:
    yaml.dump(cfg_s, f)

print('Running smoke test (2 epochs, 100 pairs)...')
r = subprocess.run(['python', 'train.py', '--config', 'config_smoke.yaml', '--ablation', 'full'])

if r.returncode != 0:
    raise RuntimeError('Smoke test FAILED')
print('\nSmoke test PASSED! Ready for full training.')

In [ ]:
# ================================================================
# CELL 7a: Config A — L1 only (baseline)
# Auto-resumes from Drive checkpoint if partially trained.
# ================================================================
import subprocess
print('Training Config A: L1 only...')
r = subprocess.run(['python', 'train.py', '--config', 'config.yaml', '--ablation', 'l1_only'])
print('Config A', 'DONE ✅' if r.returncode == 0 else 'FAILED ❌')

In [ ]:
# ================================================================
# CELL 7b: Config B — L1 + Adversarial
# Auto-resumes from Drive checkpoint if partially trained.
# ================================================================
import subprocess
print('Training Config B: L1 + Adversarial...')
r = subprocess.run(['python', 'train.py', '--config', 'config.yaml', '--ablation', 'l1_adv'])
print('Config B', 'DONE ✅' if r.returncode == 0 else 'FAILED ❌')

In [ ]:
# ================================================================
# CELL 7c: Config C — L1 + Adversarial + FFT
# Auto-resumes from Drive checkpoint if partially trained.
# ================================================================
import subprocess
print('Training Config C: L1 + Adversarial + FFT...')
r = subprocess.run(['python', 'train.py', '--config', 'config.yaml', '--ablation', 'l1_adv_fft'])
print('Config C', 'DONE ✅' if r.returncode == 0 else 'FAILED ❌')

In [ ]:
# ================================================================
# CELL 7d: Config D — Full model (MAIN SUBMISSION)
# Auto-resumes from Drive checkpoint if partially trained.
# ================================================================
import subprocess
print('Training Config D (MAIN): Full loss stack...')
r = subprocess.run(['python', 'train.py', '--config', 'config.yaml', '--ablation', 'full'])
print('Config D', 'DONE ✅' if r.returncode == 0 else 'FAILED ❌')

In [ ]:
# ================================================================
# CELL 8: Evaluate all configs on test split
# ================================================================
import os, subprocess

DRIVE_DIR = '/content/drive/MyDrive/sar2eo'

for ablation in ['l1_only', 'l1_adv', 'l1_adv_fft', 'full']:
    ckpt = f'{DRIVE_DIR}/checkpoints/{ablation}/best.pth'
    if not os.path.exists(ckpt):
        print(f'Skipping {ablation} — no checkpoint at {ckpt}')
        continue
    print(f'\n=== Evaluating {ablation} ===')
    subprocess.run([
        'python', 'eval.py',
        '--config', 'config.yaml',
        '--weights', ckpt,
        '--split', 'test'
    ])

print('\nAll evaluations done!')
print(f'Results saved to: {DRIVE_DIR}/outputs/')

In [ ]:
# ================================================================
# CELL 9: List all saved files on Drive
# ================================================================
import os

DRIVE_DIR = '/content/drive/MyDrive/sar2eo'

print('=== CHECKPOINTS ===')
for root, dirs, files in os.walk(f'{DRIVE_DIR}/checkpoints'):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path) / 1e6
        print(f'  {path.replace(DRIVE_DIR, "")} ({size:.1f} MB)')

print('\n=== OUTPUTS ===')
for root, dirs, files in os.walk(f'{DRIVE_DIR}/outputs'):
    for f in files:
        print(f'  {os.path.join(root, f).replace(DRIVE_DIR, "")}')